# Tester le RAG documentaire
Ce notebook utilise les fonctions de l’article. Exécuter les cellules dans l’ordre depuis la racine du dépôt.

Le parsing et BGE-M3 tournent localement. Le premier lancement télécharge leurs modèles et le parsing enrichi peut durer longtemps. Le routage et les réponses utilisent OpenAI avec la clé de votre `.env`. Une question documentaire transmet aussi les parents retenus et leurs images.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown, Image
from lib import ingest_document, create_workflow
from lib.context import parent_content_blocks

DATA_DIR = Path("data")
assert Path(".env").is_file(), "Copier .env.example vers .env puis renseigner sa clé."

## 1. Ajouter un document
Choisir une URL renvoyant un PDF ou le chemin d’un PDF local. Le guide est celui de l’article ; un petit PDF numérique suffit pour un premier essai. Une seconde ingestion ajoute un document à la base.

In [ ]:
source = "https://www.institutdesactuaires.com/global/gene/link.php?doc_id=17739&fg=1"
# Pour un fichier local : source = Path("mon_document.pdf")
record = ingest_document(source, DATA_DIR, title="Guide de provisionnement des sinistres en assurance non-vie")
print(f"Base enrichie : {record['parents']} parents, {record['children']} enfants.")

Le PDF, le JSON Docling, les images, les chunks et les vecteurs sont enregistrés dans `data`. Réexécuter cette ingestion pour le même PDF reprend ses fichiers déjà calculés. Aucun document n’est envoyé à OpenAI pendant l’ingestion.

## 2. Créer le workflow et tester la réponse directe
Le LLM décide si la demande exige une recherche. Une réponse directe utilise le modèle sans ouvrir la base. Le même `thread_id` conserve la conversation en RAM. Recréer le workflow après avoir ajouté un nouveau PDF.

In [ ]:
rag = create_workflow(data_dir=DATA_DIR, env_path=".env")
# Garder ce même identifiant pour les questions suivantes.
config = {"configurable": {"thread_id": "demo"}}
result = rag.invoke({"question": "Bonjour !"}, config)
display(Markdown(result["answer"]))
print("Branche :", result["route"])

## 3. Poser une question documentaire
Cette cellule utilise votre clé API. La recherche combine BM25 sur les parents et BGE-M3 sur les enfants, puis retient 3, 5 ou 7 parents avec RRF, selon le budget décidé par le LLM. La réponse cite les sections ou les objets disponibles et leurs pages.

In [ ]:
question = "Comment la méthode de Mack mesure-t-elle l'incertitude des provisions ?"
result = rag.invoke({"question": question}, config)
display(Markdown(result["answer"]))
print("Budget demandé :", result["context_k"], "parents")
print("Décision :", result["route_reason"])

## 4. Voir le contexte réellement envoyé
Ces blocs reprennent le même texte et les mêmes octets d’image que le message au modèle. Les figures suivent le texte ; ce n’est pas une reproduction de la mise en page du PDF. Aucun appel API supplémentaire.

In [ ]:
import base64
for number, parent in enumerate(result["context"]["parents"], start=1):
    display(Markdown(f"### Parent {number}"))
    for block in parent_content_blocks(parent):
        if block["type"] == "text":
            display(Markdown(block["text"]))
        else:
            encoded = block["image_url"]["url"].split(",", 1)[1]
            display(Image(data=base64.b64decode(encoded)))

## 5. Examiner les classements
L’enfant conservé pour chaque parent est son meilleur résultat dense. RRF combine les rangs des parents, pas leurs scores bruts.

In [ ]:
hits = result["retrieval"]
print("BM25, cinq premiers parents :", hits["bm25_hits"][:5])
print("Dense, cinq premiers enfants :", hits["dense_hits"][:5])
print("Parents retenus :", hits["parent_ids"])

## 6. Continuer la conversation
Le routeur utilise les trois derniers messages pour comprendre « ses limites ». Il reformule la question avant la recherche. Seules les sources de ce nouveau tour servent à la réponse. Ces trois messages comprennent les questions et les réponses, pas trois échanges complets.

Changer `thread_id` ouvre une conversation séparée. Redémarrer Python ou recréer `rag` efface cette mémoire.

In [ ]:
suite = rag.invoke({"question": "Et quelles sont ses limites ?"}, config)
display(Markdown(suite["answer"]))
print("Question autonome :", suite["search_question"])
print("Budget demandé :", suite["context_k"])
for message in suite["messages"]:
    print(message.type, ":", message.text[:200])

Une citation rend la réponse vérifiable ; elle ne garantit pas que le parsing ou l’interprétation soient corrects. Lire les pages sources, en particulier pour les chiffres et les formules. Les tests du dépôt vérifient le code, pas une performance actuarielle universelle.